In [ ]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('data/Speed Dating Data.csv', encoding='latin-1')

print(f"Dataset Shape: {df.shape}")
display(df.head())

Dataset Shape: (8378, 195)


,iid,id,gender,idg,condtn,wave,round,position,positin1,order,...,attr3_3,sinc3_3,intel3_3,fun3_3,amb3_3,attr5_3,sinc5_3,intel5_3,fun5_3,amb5_3
0,1,1.0,0,1,1,1,10,7,NaN,4,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
1,1,1.0,0,1,1,1,10,7,NaN,3,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
2,1,1.0,0,1,1,1,10,7,NaN,10,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
3,1,1.0,0,1,1,1,10,7,NaN,5,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
4,1,1.0,0,1,1,1,10,7,NaN,7,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN


These are the columns that are also in the HCMST. 

In [3]:
target_columns_dict = {
    'age': "Subject age",
    'age_o': "Partner age",
    'race': "Subject race",
    'race_o': "Partner race",  
    'gender': "Subject gender",
    'partner_gender': "Partner gender", # I CREATED THIS. 
    'attr': "Attraction",
    'Interracial_relationship': 'Interracial relationship' # I CREATED THIS 
}

### Creating a new column "interracial_relationship" with 0 for not interracial, 1 for yes interracial
this format matches what is in the HCMST dataset

Checking if it is a interracial relationship. i.e: do their races match?



In [4]:
df['Interracial_relationship'] = (df['race'] != df['race_o']).astype('Int64')
df[['race','race_o', 'Interracial_relationship']]

,race,race_o,Interracial_relationship
0,4.0,2.0,1
1,4.0,2.0,1
2,4.0,4.0,0
3,4.0,2.0,1
4,4.0,3.0,1
...,...,...,...
8373,2.0,3.0,1
8374,2.0,6.0,1
8375,2.0,3.0,1
8376,2.0,4.0,1


In [ ]:
race_mapping = {
    1: 'BLACK',
    2: 'WHITE',
    3: 'HISPANIC',
    4: 'ASIAN',
    5: 'NATIVE AMERICAN',
    6: 'OTHER'
}

# 2. Apply the mapping to the column
df["race"] = df["race"].map(race_mapping)
df["race_o"] = df["race_o"].map(race_mapping)

### Create a partner_gender column

this dataset does not have it.

In [6]:
df['partner_gender'] = np.where(df['gender'] == 0, 1, 0)

df[['gender','partner_gender']]

,gender,partner_gender
0,0,1
1,0,1
2,0,1
3,0,1
4,0,1
...,...,...
8373,1,0
8374,1,0
8375,1,0
8376,1,0


Creating new df with only the columns that are also in HCMST dataset

In [7]:
columbia_filtered = df[target_columns_dict.keys()].copy()

In [8]:
columbia_filtered

,age,age_o,race,race_o,gender,partner_gender,attr,Interracial_relationship
0,21.0,27.0,ASIAN,WHITE,0,1,6.0,1
1,21.0,22.0,ASIAN,WHITE,0,1,7.0,1
2,21.0,22.0,ASIAN,ASIAN,0,1,5.0,0
3,21.0,23.0,ASIAN,WHITE,0,1,7.0,1
4,21.0,24.0,ASIAN,HISPANIC,0,1,5.0,1
...,...,...,...,...,...,...,...,...
8373,25.0,26.0,WHITE,HISPANIC,1,0,3.0,1
8374,25.0,24.0,WHITE,OTHER,1,0,4.0,1
8375,25.0,29.0,WHITE,HISPANIC,1,0,4.0,1
8376,25.0,22.0,WHITE,ASIAN,1,0,4.0,1


### Rename Columns to match HCTMS

In [9]:
columbia_filtered.rename(columns=target_columns_dict, inplace=True)

### Drop NA

In [10]:
columbia_filtered = columbia_filtered.dropna()

### Convert to INT64

In [11]:
columbia_filtered

,Subject age,Partner age,Subject race,Partner race,Subject gender,Partner gender,Attraction,Interracial relationship
0,21.0,27.0,ASIAN,WHITE,0,1,6.0,1
1,21.0,22.0,ASIAN,WHITE,0,1,7.0,1
2,21.0,22.0,ASIAN,ASIAN,0,1,5.0,0
3,21.0,23.0,ASIAN,WHITE,0,1,7.0,1
4,21.0,24.0,ASIAN,HISPANIC,0,1,5.0,1
...,...,...,...,...,...,...,...,...
8373,25.0,26.0,WHITE,HISPANIC,1,0,3.0,1
8374,25.0,24.0,WHITE,OTHER,1,0,4.0,1
8375,25.0,29.0,WHITE,HISPANIC,1,0,4.0,1
8376,25.0,22.0,WHITE,ASIAN,1,0,4.0,1


In [12]:
# 1. Get a list of all columns except 'race' and 'race_o'
cols_to_convert = columbia_filtered.columns.drop(['Subject race', 'Partner race'])

# 2. Convert only those specific columns to int64
columbia_filtered[cols_to_convert] = columbia_filtered[cols_to_convert].round().astype('Int64')

C:\Users\new user\AppData\Local\Temp\ipykernel_29828\3879968598.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  columbia_filtered[cols_to_convert] = columbia_filtered[cols_to_convert].round().astype('Int64')


In [13]:
#cols_to_convert = columbia_filtered.columns.drop(['race', 'race_o'])
#columbia_filtered = columbia_filtered.astype('int64')

### Remove rows where attraction is 0

In [14]:
# Find the index of rows where Attraction is 0, then drop them
index_to_drop = columbia_filtered[columbia_filtered['Attraction'] == 0].index
columbia_filtered = columbia_filtered.drop(index_to_drop)

Normalize Attraction: 

In [15]:
columbia_filtered

,Subject age,Partner age,Subject race,Partner race,Subject gender,Partner gender,Attraction,Interracial relationship
0,21,27,ASIAN,WHITE,0,1,6,1
1,21,22,ASIAN,WHITE,0,1,7,1
2,21,22,ASIAN,ASIAN,0,1,5,0
3,21,23,ASIAN,WHITE,0,1,7,1
4,21,24,ASIAN,HISPANIC,0,1,5,1
...,...,...,...,...,...,...,...,...
8373,25,26,WHITE,HISPANIC,1,0,3,1
8374,25,24,WHITE,OTHER,1,0,4,1
8375,25,29,WHITE,HISPANIC,1,0,4,1
8376,25,22,WHITE,ASIAN,1,0,4,1


### Export 

This file will be used/loaded in combining_datasets notebook

In [16]:
columbia_filtered.to_csv('Columbia_filtered.csv', index=False)

### Running the same steps for the entire dataset

In [ ]:

df.rename(columns=target_columns_dict, inplace=True)
columbia_filtered = columbia_filtered.dropna()
cols_to_convert = columbia_filtered.columns.drop(['Subject race', 'Partner race'])
columbia_filtered[cols_to_convert] = columbia_filtered[cols_to_convert].round().astype('Int64')
